# Adversarial Search: Playing Dots and Boxes


## Instructions

Total Points: Undegraduates 100, graduate students 110

Complete this notebook and submit it. The notebook needs to be a complete project report with your implementation, documentation including a short discussion of how your implementation works and your design choices, and experimental results (e.g., tables and charts with simulation results) with a short discussion of what they mean. Use the provided notebook cells and insert additional code and markdown cells as needed.

## Introduction

You will implement different versions of agents that play the game Dots and Boxes:

> "Dots and Boxes is a pencil-and-paper game for two players. The game starts with an empty grid of dots. Usually two players take turns adding a single horizontal or vertical line between two unjoined adjacent dots. A player who completes the fourth side of a 1x1 box earns one point and takes another turn. A point is typically recorded by placing a mark that identifies the player in the box, such as an initial. The game ends when no more lines can be placed. The winner is the player with the most points. The board may be of any size grid." (see [Dots and Boxes on Wikipedia](https://en.wikipedia.org/wiki/Dots_and_Boxes))

You can play Dots and Boxes [here](https://www.math.ucla.edu/~tom/Games/dots&boxes.html).

## Task 1: Defining the Search Problem [10 point]

Define the components of the search problem associated with this game:

* Initial state
* Actions
* Transition model
* Test for the terminal state
* Utility for terminal states

1. Trạng thái ban đầu (Initial State):

Bàn cờ gồm $$r \times c$$ điểm, chưa có cạnh nào được vẽ và chưa có ô vuông nào thuộc về người chơi. Người chơi +1 được đi trước.

Trạng thái được biểu diễn bằng dictionary gồm:
- size: số hàng và cột điểm.
- lines: tập hợp các cạnh đã được vẽ.
- boxes: các ô vuông đã hoàn thành và người sở hữu.
- turn: người chơi đang đến lượt.

2. Hành động (Actions):

Người chơi chọn một cạnh ngang hoặc dọc nối hai điểm liền kề, với điều kiện cạnh đó chưa được vẽ.

3. Mô hình chuyển trạng thái (Transition Model):

Sau khi thực hiện một hành động, cạnh tương ứng được thêm vào bàn cờ. Nếu hành động hoàn thành một hoặc nhiều ô vuông, người chơi được cộng điểm và tiếp tục lượt. Ngược lại, lượt chơi chuyển cho đối thủ.

4. Trạng thái kết thúc (Terminal State):

Trò chơi kết thúc khi tất cả các cạnh hợp lệ trên bàn cờ đã được vẽ.

5. Hàm lợi ích (Utility Function):

Hàm utility được tính bằng hiệu số ô vuông mà hai người chơi sở hữu:

$$U(s)=Score(+1)-Score(-1)$$
- U(s)>0: người chơi +1 thắng.
- U(s)=0: hai người chơi hòa.
- U(s)<0: người chơi -1 thắng.

How big is the state space? Give an estimate and explain it.

Với bàn cờ gồm r hàng và c cột điểm, tổng số cạnh có thể vẽ là:
$$E=r(c-1)+c(r-1)$$

Trong đó:

- r(c-1): số cạnh ngang.
- c(r-1): số cạnh dọc.
- E: tổng số cạnh.

Mỗi cạnh có hai trạng thái: đã vẽ hoặc chưa vẽ. Vì vậy, số cấu hình cạnh có thể có là:
$$S_{\text{edges}}=2^E$$
Ví dụ:
- Bàn 2 × 3 điểm có 7 cạnh, tương ứng 2^7=128 cấu hình cạnh.
- Bàn 4 × 4 điểm có 24 cạnh, tương ứng 2^24=16.777.216 cấu hình cạnh.

Tuy nhiên, đây chỉ là số cấu hình cạnh, chưa phải toàn bộ không gian trạng thái của trò chơi. Một trạng thái đầy đủ còn phải lưu người sở hữu các ô vuông và người đang đến lượt.

Do đó, không gian trạng thái thực tế lớn hơn và tăng rất nhanh khi kích thước bàn cờ tăng.

How big is the game tree that minimax search will go through? Give an estimate and explain it.

Giả sử bàn cờ có (E) cạnh. Ở trạng thái ban đầu, người chơi có E nước đi hợp lệ. Sau mỗi lượt, số cạnh chưa được vẽ giảm đi một.

Do đó, số trình tự nước đi hoàn chỉnh có thể lên đến:
$$L=E!$$

Ví dụ, bàn cờ 4 × 4 điểm có 24 cạnh:
$$L=24!\approx6,20\times10^{23}$$

Đây là số lượng trình tự nước đi hoàn chỉnh trong cây tìm kiếm chưa cắt tỉa, không phải số trạng thái phân biệt.

Như vậy, cây trò chơi tăng theo cấp giai thừa khi kích thước bàn cờ tăng. Điều này khiến Minimax tìm kiếm đầy đủ trở nên rất tốn thời gian và bộ nhớ.

Alpha-Beta Pruning có thể giảm số nút cần duyệt bằng cách loại bỏ những nhánh không ảnh hưởng đến quyết định tối ưu, nhưng không loại bỏ được hoàn toàn vấn đề bùng nổ không gian tìm kiếm.

## Task 2: Game Environment and Random Agent [30 point]

You need to think about a data structure to represent the board meaning he placed lines and who finished what box. There are many options. Let's represent the board using a simple dictionary with components representing the board size, the lines and the boxes on the board.

**Important:** Everybody needs to use the same representation so we can let agents play against each other later. 

In [1]:
board = {
    'size': (4, 4),  ### number of rows and columns of dots
    'lines': dict(), ### keys are the set of drawn lines
    'boxes': dict    ### keys are the boxes and the value is the player who completed each box
}

def draw_line(board, orientation, row, col):
    """
    Place a line on an exiting board.
       
    Parameters
    ----------
    board: dict
        the board
    orientation: str
        either 'h' or 'v' for horizontal or vertical
    row, col: int
        index of the starting dot for the line (starting with 0)
    
    """
    
    if orientation not in ['h', 'v']:
        return False
        
    if row < 0 or col < 0:
        return False
        
    if row >= board['size'][0] + (orientation == 'v') or col >= board['size'][1] + (orientation == 'h'):
        return False
        
    if (orientation, row, col) in board['lines']:
        return False
            
    board["lines"][(orientation, row, col)] = True
    return True
    

print(draw_line(board, "h", 1, 1))
print(draw_line(board, "v", 1, 1))

# this should not work
print(draw_line(board, "h", 1, 1))

board

True
True
False


{'size': (4, 4),
 'lines': {('h', 1, 1): True, ('v', 1, 1): True},
 'boxes': dict}

In [ ]:
import random
import time
import pandas as pd

from copy import deepcopy
from math import inf
from IPython.display import display


# 1. Khởi tạo bàn cờ mới
def new_board(rows=3, cols=3):
    if rows < 2 or cols < 2:
        raise ValueError(
            "Board must have at least 2 rows and 2 columns"
        )

    return {
        'size': (rows, cols),
        'lines': {},
        'boxes': {},
        'turn': 1
    }


# 2. Liệt kê tất cả các cạnh hợp lệ
def all_edges(board):
    rows, cols = board['size']

    horizontal = [
        ('h', r, c)
        for r in range(rows)
        for c in range(cols - 1)
    ]

    vertical = [
        ('v', r, c)
        for r in range(rows - 1)
        for c in range(cols)
    ]

    return horizontal + vertical


# 3. Bổ sung kiểm tra tính hợp lệ của cạnh
def draw_line(board, orientation, row, col):
    action = (orientation, row, col)

    if action not in all_edges(board):
        return False

    if action in board['lines']:
        return False

    board['lines'][action] = True
    return True


# 4. Chuẩn hóa bàn cờ mẫu 
board = {
    'size': board['size'],
    'lines': board['lines'].copy(),
    'boxes': {},
    'turn': 1
}

print("Setup completed successfully")
print("Total edges:", len(all_edges(board)))
print("Drawn edges:", len(board['lines']))

Setup completed successfully
Total edges: 24
Drawn edges: 2


Write code to display the board. **Bonus point: Post your visualization code with an example output to the discussion board. The best visualization will earn you bonus participation points in this class.**

In [3]:
def show_board(board):
    rows,cols=board['size']
    for r in range(rows):
        top = ''
        for c in range(cols):
            top += '•'
            if c < cols-1: top += '───' if ('h',r,c) in board['lines'] else '   '
        print(top)
        if r<rows-1:
            middle = ''
            for c in range(cols):
                middle += '│' if ('v',r,c) in board['lines'] else ' '
                if c < cols-1: middle += f' {board["boxes"].get((r,c)," "):>2}' if (r,c) in board['boxes'] else '   '
            print(middle)
    print('Score:',{p:sum(v==p for v in board['boxes'].values()) for p in (1,-1)},'Turn:',board['turn'])

show_board(board)

•   •   •   •
             
•   •───•   •
    │        
•   •   •   •
             
•   •   •   •
Score: {1: 0, -1: 0} Turn: 1


Implement helper functions for:

* The transition model $result(s, a)$.
* The utility function $utility(s)$.
* Check for terminal states $terminal(s)$.
* A check for available actions in each state $actions(s)$.

__Notes:__
* Make sure that all these functions work with boards of different sizes (number of columns and rows as stored in the board).
* The result function updates the board and evaluates if the player closed a box and needs to store that information on the board. Add elements of the form `(row,col): player` to the board dictionary. `row` and `col` are the coordinates for the box and `player` is +1 or -1 representing the player. For example `(0,0): -1` means that the top-left box belongs to the other player. 
* _Important:_ Remember that a player goes again after she completes a box!

In [ ]:
def actions(board):
    return [e for e in all_edges(board) if e not in board['lines']]

def box_edges(r,c):
    return (('h',r,c),('h',r+1,c),('v',r,c),('v',r,c+1))

def result(board, action):
    if action not in actions(board):
        raise ValueError(f'Illegal edge: {action}')
    nxt = {'size':board['size'], 'lines':board['lines'].copy(),
           'boxes':board['boxes'].copy(), 'turn':board['turn']}
    nxt['lines'][action]=True
    completed=0
    rows,cols=nxt['size']
    for r in range(rows-1):
        for c in range(cols-1):
            if (r,c) not in nxt['boxes'] and all(e in nxt['lines'] for e in box_edges(r,c)):
                nxt['boxes'][(r,c)]=board['turn']
                completed+=1
    if completed==0: nxt['turn']=-board['turn']
    return nxt

def terminal(board):
    return len(board['lines'])==len(all_edges(board))

def utility(board,player=1):
    if not terminal(board): return None
    return sum(v==player for v in board['boxes'].values())-sum(v==-player for v in board['boxes'].values())

# Kiểm tra luật đặc biệt: hoàn thành ô vuông thì được đi thêm lượt
b=new_board(2,2)
for e in [('h',0,0),('h',1,0),('v',0,0)]: b=result(b,e)
assert b['turn']==-1
b=result(b,('v',0,1))
assert b['boxes'][(0,0)]==-1 and b['turn']==-1
assert terminal(b) and utility(b,-1)==1
print('Transition / extra-turn / terminal / utility checks passed')

Transition / extra-turn / terminal / utility checks passed


Implement an agent that plays randomly. Make sure the agent function receives as the percept the board and returns a valid action. Use an agent function definition with the following signature (arguments):

`def random_player(board, player = None): ...`

The argument `player` is used for agents that do not store what side they are playing. The value passed on by the environment should be 1 ot -1 for playerred and yellow, respectively.  See [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) for an example.

In [5]:
def random_player(board, player=None):
    return random.choice(actions(board))

def play_game(agent_plus,agent_minus,rows=3,cols=3,initial=None):
    board=new_board(rows,cols) if initial is None else deepcopy(initial)
    while not terminal(board):
        agent=agent_plus if board['turn']==1 else agent_minus
        move=agent(board,board['turn'])
        board=result(board,move)
    return board

def tournament(agent_plus,agent_minus,games=100,rows=3,cols=3):
    outcomes={'+1':0,'-1':0,'draw':0}
    for _ in range(games):
        end=play_game(agent_plus,agent_minus,rows,cols)
        u=utility(end,1)
        outcomes['+1' if u>0 else '-1' if u<0 else 'draw']+=1
    return outcomes
print('Sample valid random action:',random_player(new_board(2,3)))

Sample valid random action: ('h', 1, 1)


Let two random agents play against each other 1000 times. Look at the [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) to see how the environment uses the agent functions to play against each other.

How often does each player win? Is the result expected?

In [6]:
random.seed(2026)
random_results=tournament(random_player,random_player,games=1000,rows=3,cols=3)
print('Random vs Random:',random_results)
print('Rates:', {k:round(v/1000,3) for k,v in random_results.items()})

Random vs Random: {'+1': 411, '-1': 413, 'draw': 176}
Rates: {'+1': 0.411, '-1': 0.413, 'draw': 0.176}


**Discussion** 

Sau 1.000 ván đấu giữa hai Random Agent trên bàn cờ 3 × 3 điểm, người chơi +1 thắng 411 ván (41,1%), người chơi -1 thắng 413 ván (41,3%) và có 176 ván hòa (17,6%).

Kết quả cho thấy tỷ lệ thắng của hai người chơi gần tương đương nhau, do cả hai đều lựa chọn nước đi ngẫu nhiên và không có chiến lược tối ưu.

Sự chênh lệch nhỏ về số trận thắng có thể xuất phát từ tính ngẫu nhiên của quá trình chọn nước đi. Trong thực nghiệm này, chưa có bằng chứng rõ ràng cho thấy người đi trước có lợi thế đáng kể.

## Task 3: Minimax Search with Alpha-Beta Pruning [30 points]

### Implement the search starting.

Implement the search starting from a given board and specifying the player and put it into an agent function.
You can use code from the [tic-tac-toe example](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_alpha_beta_tree_search.ipynb).

__Notes:__ 
* Make sure that all your agent functions have a signature consistent with the random agent above.
* The search space for larger board may be too large. You can experiment with smaller boards.
* Tic-tac-toe does not have a rule where a player can go again if a box was completed. You need to adapt the tree search to reflect that rule.

In [7]:
def immediate_boxes(board,action):
    nxt=result(board,action)
    return len(nxt['boxes'])-len(board['boxes'])

def ordered_actions(board,ordering=False):
    moves=actions(board)
    if ordering:
        moves.sort(key=lambda a: immediate_boxes(board,a),reverse=True)
    return moves

def alpha_beta_search(board,player=None,cutoff=None,ordering=False,heuristic=None):
    """Return move/value/nodes/seconds. MAX is root player, even after extra turns."""
    root=board['turn'] if player is None else player
    if root != board['turn']: raise ValueError('player must equal board[turn]')
    nodes=0
    start=time.perf_counter()

    def visit(state,depth,alpha,beta):
        nonlocal nodes
        nodes+=1
        if terminal(state): return utility(state,root)
        if cutoff is not None and depth>=cutoff:
            return heuristic(state,root) if heuristic else heuristic_eval(state,root)
        maximize=state['turn']==root
        if maximize:
            best=-inf
            for a in ordered_actions(state,ordering):
                best=max(best,visit(result(state,a),depth+1,alpha,beta))
                alpha=max(alpha,best)
                if alpha>=beta: break
            return best
        best=inf
        for a in ordered_actions(state,ordering):
            best=min(best,visit(result(state,a),depth+1,alpha,beta))
            beta=min(beta,best)
            if alpha>=beta: break
        return best

    if terminal(board): return {'move':None,'value':utility(board,root),'nodes':1,'seconds':0}
    best=-inf
    best_move=None
    alpha,beta=-inf,inf
    for a in ordered_actions(board,ordering):
        val=visit(result(board,a),1,alpha,beta)
        if val>best: best,best_move=val,a
        alpha=max(alpha,best)
    return {'move':best_move,'value':best,'nodes':nodes,'seconds':time.perf_counter()-start}

def alpha_beta_player(board,player=None):
    return alpha_beta_search(board,player,ordering=True)['move']

print(alpha_beta_search(new_board(2,2)))

{'move': ('h', 0, 0), 'value': -1, 'nodes': 30, 'seconds': 0.0002545001916587353}


Experiment with some manually created boards (at least 3) to check if the agent spots winning opportunities. Discuss the results.

In [ ]:
# Trường hợp 1: Chỉ còn thiếu một cạnh để hoàn thành ô
b1=new_board(2,2)
for a in [('h',0,0),('h',1,0),('v',0,0)]: b1=result(b1,a)
# Trường hợp 2: Một cạnh có thể hoàn thành đồng thời hai ô
b2=new_board(2,3)
for a in [('h',0,0),('h',0,1),('h',1,0),('h',1,1),('v',0,0),('v',0,2)]: b2=result(b2,a)
# Trường hợp 3: Có cơ hội hoàn thành ô và nhiều nước đi hợp lệ khác
b3=new_board(2,3)
for a in [('h',0,0),('h',1,0),('v',0,0)]: b3=result(b3,a)
manual_boards=[('One missing edge',b1),('Two boxes at once',b2),('One box available',b3)]
for label,b in manual_boards:
    result_ab=alpha_beta_search(b,ordering=True)
    print(label,result_ab)
    show_board(b)
assert alpha_beta_search(b1)['move']==('v',0,1)
assert alpha_beta_search(b2)['move']==('v',0,1)

One missing edge {'move': ('v', 0, 1), 'value': 1, 'nodes': 1, 'seconds': 3.8499943912029266e-05}
•───•
│    
•───•
Score: {1: 0, -1: 0} Turn: -1
Two boxes at once {'move': ('v', 0, 1), 'value': 2, 'nodes': 1, 'seconds': 3.760005347430706e-05}
•───•───•
│       │
•───•───•
Score: {1: 0, -1: 0} Turn: 1
One box available {'move': ('v', 0, 1), 'value': 2, 'nodes': 24, 'seconds': 0.0003846001345664263}
•───•   •
│        
•───•   •
Score: {1: 0, -1: 0} Turn: -1


**Discussion** 

Qua ba trạng thái kiểm thử thủ công, thuật toán Alpha-Beta đều lựa chọn nước đi ('v', 0, 1).

Ở trạng thái thứ nhất, nước đi này hoàn thành ô vuông cuối cùng và mang lại giá trị utility +1.

Ở trạng thái thứ hai, cạnh được chọn đồng thời hoàn thành hai ô vuông, mang lại giá trị utility +2.

Ở trạng thái thứ ba, thuật toán nhận diện cơ hội ghi điểm và lựa chọn nước đi dẫn tới giá trị utility tối ưu +2 khi tiếp tục tìm kiếm đầy đủ.

Kết quả cho thấy Alpha-Beta có khả năng nhận diện những nước đi mang lại lợi thế, đồng thời xử lý đúng luật hoàn thành ô và được tiếp tục lượt chơi.

How long does it take to make a move? Start with a smaller board make the board larger. What is the largest board you can solve?

In [9]:
rows=[]
for r,c in [(2,2),(2,3),(2,4)]:
    b=new_board(r,c)
    ans=alpha_beta_search(b,ordering=True)
    rows.append({'Dots':f'{r}x{c}','Edges':len(all_edges(b)),**ans})
full_depth_df=pd.DataFrame(rows)
display(full_depth_df)

,Dots,Edges,move,value,nodes,seconds
0,2x2,4,"(h, 0, 0)",-1,30,0.000696
1,2x3,7,"(v, 0, 1)",0,896,0.014854
2,2x4,10,"(h, 0, 0)",-1,17631,0.441272


**Discussion**

Thực nghiệm tìm kiếm đầy đủ được thực hiện trên các bàn cờ từ 2 × 2 đến 2 × 4 điểm.

Khi tăng kích thước từ 2 × 2 lên 2 × 4 điểm, số nút duyệt tăng từ 30 lên 17.631, đồng thời thời gian xử lý tăng từ 0,000696 giây lên 0,441272 giây. Điều này cho thấy chi phí tìm kiếm tăng nhanh khi số lượng cạnh trên bàn cờ tăng.

Nguyên nhân là khi số cạnh tăng, số lượng hành động hợp lệ và số nhánh trong cây trò chơi cũng tăng nhanh.

Trong phạm vi thực nghiệm này, bàn lớn nhất được kiểm tra bằng Alpha-Beta đầy đủ là 2 × 4 điểm, với 10 cạnh. Đây không phải giới hạn tối đa tuyệt đối của thuật toán hoặc máy tính, mà chỉ là kích thước lớn nhất đã được thử nghiệm.

Kết quả cho thấy việc tìm kiếm đầy đủ ngày càng tốn kém khi kích thước bàn cờ tăng. Vì vậy, cần kết hợp giới hạn độ sâu và hàm đánh giá heuristic để xử lý các bàn lớn hơn.

### Move ordering

Starting the search with better moves will increase the efficiency of alpha-beta pruning. Describe and implement a simple move ordering strategy. 

Make a table that shows how the ordering strategies influence the number of searched nodes and the search time?

In [10]:
ordering_rows=[]
for label,b in manual_boards+[('Empty 2x3',new_board(2,3)),('Empty 2x4',new_board(2,4))]:
    for use_order in (False,True):
        x=alpha_beta_search(b,ordering=use_order)
        ordering_rows.append({'Board':label,'Ordering':use_order,**x})
ordering_df=pd.DataFrame(ordering_rows)
display(ordering_df)

,Board,Ordering,move,value,nodes,seconds
0,One missing edge,False,"(v, 0, 1)",1,1,0.000095
1,One missing edge,True,"(v, 0, 1)",1,1,0.000084
2,Two boxes at once,False,"(v, 0, 1)",2,1,0.000061
3,Two boxes at once,True,"(v, 0, 1)",2,1,0.000086
4,One box available,False,"(v, 0, 1)",2,43,0.001458
5,One box available,True,"(v, 0, 1)",2,24,0.001549
6,Empty 2x3,False,"(v, 0, 1)",0,1397,0.024072
7,Empty 2x3,True,"(v, 0, 1)",0,896,0.013570
8,Empty 2x4,False,"(h, 0, 0)",-1,49139,0.550206
9,Empty 2x4,True,"(h, 0, 0)",-1,17631,0.386598


**Discussion**

Sau khi áp dụng Move ordering, thuật toán Alpha-Beta ưu tiên các nước đi hoàn thành nhiều ô vuông nhất trước khi xét những nước đi khác.

Trên bàn cờ trống 2 × 3 điểm, số nút giảm từ 1.397 xuống 896, tương đương khoảng 35,86%.

Đặc biệt, trên bàn cờ trống 2 × 4 điểm, số nút giảm từ 49.139 xuống còn 17.631 (giảm khoảng 64,12%), đồng thời thời gian chạy giảm từ 0,550206 giây xuống 0,386598 giây.

Nguyên nhân là các nước đi có giá trị tốt được xét sớm giúp cập nhật giới hạn alpha và beta nhanh hơn, từ đó tăng khả năng loại bỏ các nhánh không ảnh hưởng đến quyết định tối ưu.

Tuy nhiên, hiệu quả còn phụ thuộc vào trạng thái bàn cờ. Với những trạng thái chỉ còn một nước đi hợp lệ, Move ordering không làm giảm số nút. Ngoài ra, việc tính độ ưu tiên cũng phát sinh chi phí xử lý nên thời gian chạy không nhất thiết giảm trong mọi trường hợp.

Nhìn chung, Move ordering cải thiện đáng kể hiệu suất Alpha-Beta trong các trạng thái thử nghiệm có nhiều nước đi hợp lệ.

### The first few moves

Start with an empty board. This is the worst case scenario for minimax search with alpha-beta pruning since it needs solve all possible games that can be played (minus some pruning) before making the decision. What can you do? 

In [ ]:
# Hàm đánh giá heuristic dựa trên điểm số và ô có ba cạnh
def heuristic_eval(board,player=1):
    score=sum(v==player for v in board['boxes'].values())-sum(v==-player for v in board['boxes'].values())
    danger=0
    for r in range(board['size'][0]-1):
        for c in range(board['size'][1]-1):
            if (r,c) not in board['boxes']:
                n=sum(e in board['lines'] for e in box_edges(r,c))
                if n==3: danger+=1
    return score + (0.4*danger if board['turn']==player else -0.4*danger)

def quick_opening_player(board,player=None):
    # Ưu tiên nước đi ghi điểm hoặc cạnh an toàn
    # nếu không có, sử dụng tìm kiếm heuristic giới hạn độ sâu
    safe=[]
    for a in actions(board):
        future=result(board,a)
        if immediate_boxes(board,a)>0: return a
        if all(sum(e in future['lines'] for e in box_edges(r,c))<3
               for r in range(board['size'][0]-1)
               for c in range(board['size'][1]-1)
               if (r,c) not in future['boxes']):
            safe.append(a)
    return safe[0] if safe else alpha_beta_search(board,cutoff=3,ordering=True)['move']
print('Practical first move:',quick_opening_player(new_board(4,4)))

Practical first move: ('h', 0, 0)


**Discussion**

Ở trạng thái bàn cờ trống, số nước đi hợp lệ lớn nhất nên cây tìm kiếm có số lượng nhánh rất lớn. Vì vậy, việc sử dụng Alpha-Beta đầy đủ để xác định nước đi đầu tiên có thể tốn nhiều thời gian.

Để cải thiện, có thể sử dụng chiến lược chọn nước đi nhanh kết hợp với tìm kiếm giới hạn độ sâu.

Trong chương trình, hàm quick_opening_player() ưu tiên:
1. Chọn nước đi hoàn thành ô vuông nếu có thể.
2. Nếu không, chọn một cạnh an toàn, tránh tạo ô vuông có ba cạnh để đối thủ dễ dàng ghi điểm ở lượt kế tiếp.
3. Nếu không có nước đi an toàn, sử dụng Heuristic Alpha-Beta với cutoff = 3.

Trên bàn 4 × 4 điểm trống, thuật toán thử nghiệm chọn cạnh ('h', 0, 0).

Cách tiếp cận này giúp giảm chi phí tính toán ở giai đoạn đầu, nhưng không bảo đảm nước đi được chọn luôn tối ưu.

### Playtime

Let the Minimax Search agent play a random agent on a small board. Analyze wins, losses and draws.

In [ ]:
# Sử dụng bàn nhỏ 2x3 điểm để tìm kiếm đầy đủ với chi phí thấp
random.seed(2026)
results_ab=tournament(alpha_beta_player,random_player,games=30,rows=2,cols=3)
print('Alpha-Beta (+1) vs Random (-1):',results_ab)

Alpha-Beta (+1) vs Random (-1): {'+1': 0, '-1': 0, 'draw': 30}


**Discussion**

Trong 30 ván thi đấu giữa Alpha-Beta (+1) và Random Agent (-1) trên bàn 2 × 3 điểm, tất cả các trận đều kết thúc với kết quả hòa.

Nguyên nhân là bàn cờ 2 × 3 điểm chỉ có hai ô vuông. Theo kết quả tìm kiếm đầy đủ, người chơi +1 không thể bảo đảm chiến thắng nếu đối thủ có những lựa chọn tốt. Alpha-Beta vì vậy lựa chọn các nước đi nhằm bảo đảm giá trị utility tối ưu là 0.

Kết quả 30 trận hòa cho thấy thuật toán duy trì được kết quả tối ưu trong điều kiện kiểm thử này. Tuy nhiên, do bàn cờ nhỏ và số trận thử nghiệm còn hạn chế, cần kiểm tra thêm các kích thước bàn khác để đánh giá khả năng thi đấu của tác tử.

## Task 4: Heuristic Alpha-Beta Tree Search [30 points] 

### Heuristic evaluation function

Define and implement a heuristic evaluation function.

In [13]:
for label,b in manual_boards:
    print(label,'h(+1)=',heuristic_eval(b,1),'h(-1)=',heuristic_eval(b,-1))

One missing edge h(+1)= -0.4 h(-1)= 0.4
Two boxes at once h(+1)= 0.8 h(-1)= -0.8
One box available h(+1)= -0.4 h(-1)= 0.4


### Cutting off search 

Modify your Minimax Search with Alpha-Beta Pruning to cut off search at a specified depth and use the heuristic evaluation function. Experiment with different cutoff values.

In [14]:
def heuristic2_player(board,player=None):
    return alpha_beta_search(board,player,cutoff=2,ordering=True)['move']

def heuristic4_player(board,player=None):
    return alpha_beta_search(board,player,cutoff=4,ordering=True)['move']

cutoff_rows=[]
for label,b in manual_boards+[('Empty 2x3',new_board(2,3))]:
    for cutoff in (2,4,None):
        ans=alpha_beta_search(b,cutoff=cutoff,ordering=True)
        cutoff_rows.append({'Board':label,'Cutoff':str(cutoff),**ans})
cutoff_df=pd.DataFrame(cutoff_rows)
display(cutoff_df)

,Board,Cutoff,move,value,nodes,seconds
0,One missing edge,2,"(v, 0, 1)",1.0,1,0.000060
1,One missing edge,4,"(v, 0, 1)",1.0,1,0.000028
2,One missing edge,None,"(v, 0, 1)",1.0,1,0.000048
3,Two boxes at once,2,"(v, 0, 1)",2.0,1,0.000037
4,Two boxes at once,4,"(v, 0, 1)",2.0,1,0.000032
5,Two boxes at once,None,"(v, 0, 1)",2.0,1,0.000031
6,One box available,2,"(v, 0, 1)",1.0,10,0.000352
7,One box available,4,"(v, 0, 1)",2.0,24,0.001104
8,One box available,None,"(v, 0, 1)",2.0,24,0.000359
9,Empty 2x3,2,"(h, 0, 0)",0.0,19,0.000402


**Discussion**

Qua thực nghiệm, Heuristic Alpha-Beta với cutoff = 2 duyệt ít nút hơn so với cutoff = 4 và Alpha-Beta đầy đủ.

Đối với bàn cờ trống 2 × 3 điểm, cutoff = 2 chỉ duyệt 19 nút, cutoff = 4 duyệt 209 nút, trong khi tìm kiếm đầy đủ duyệt 896 nút.

Về chất lượng đánh giá, cutoff = 2 có thể cho giá trị chưa chính xác vì chỉ quan sát được một phần cây tìm kiếm. Ví dụ, ở trạng thái có một ô sẵn sàng được hoàn thành, cutoff = 2 trả về giá trị 1, trong khi cutoff = 4 và tìm kiếm đầy đủ trả về giá trị 2.

Trên bàn cờ trống 2 × 3 điểm, cutoff = 2 chọn cạnh ('h', 0, 0), còn cutoff = 4 và Full chọn ('v', 0, 1). Điều này cho thấy thay đổi độ sâu có thể ảnh hưởng đến nước đi được chọn.

Tuy nhiên, giá trị heuristic khác với giá trị Minimax chính xác, nên không thể chỉ dựa vào giá trị ước lượng để kết luận nước đi nào tốt hơn.

Nhìn chung, cutoff nhỏ làm giảm số nút duyệt và thời gian xử lý nhưng tăng sự phụ thuộc vào hàm heuristic. Cutoff lớn giúp tìm kiếm sâu hơn, đổi lại chi phí tính toán tăng lên.

How many nodes are searched and how long does it take to make a move? Start with a smaller board with 4 columns and make the board larger by adding columns.

In [15]:
scale_rows=[]
for r,c in [(3,4),(3,5),(3,6)]:
    b=new_board(r,c)
    for cutoff in (2,4):
        ans=alpha_beta_search(b,cutoff=cutoff,ordering=True)
        scale_rows.append({'Dots':f'{r}x{c}','Edges':len(all_edges(b)),'Cutoff':cutoff,**ans})
scale_df=pd.DataFrame(scale_rows)
display(scale_df)

,Dots,Edges,Cutoff,move,value,nodes,seconds
0,3x4,17,2,"(h, 0, 0)",0.0,49,0.005954
1,3x4,17,4,"(h, 0, 0)",0.0,803,0.058200
2,3x5,22,2,"(h, 0, 0)",0.0,64,0.006470
3,3x5,22,4,"(h, 0, 0)",0.0,1368,0.170842
4,3x6,27,2,"(h, 0, 0)",0.0,79,0.015283
5,3x6,27,4,"(h, 0, 0)",0.0,2083,0.321727


**Discussion**

Khi tăng kích thước bàn cờ từ 3 × 4 lên 3 × 6 điểm, số nút duyệt và thời gian xử lý đều tăng ở cả hai cấu hình cutoff.

Với cutoff = 2, số nút tăng từ 49 lên 79 và thời gian xử lý tăng từ 0,005954 giây lên 0,015283 giây khi mở rộng bàn cờ từ 3 × 4 lên 3 × 6 điểm.

Trong khi đó, với cutoff = 4, số nút tăng từ 803 lên 2.083 và thời gian xử lý tăng từ 0,058200 giây lên 0,321727 giây.

Kết quả cho thấy khi tăng độ sâu tìm kiếm, chi phí tính toán tăng rõ rệt, đặc biệt khi số lượng cạnh và các nước đi hợp lệ trên bàn cờ lớn hơn.

Tuy nhiên, nhờ giới hạn độ sâu và sử dụng heuristic, thuật toán vẫn có thể xử lý các bàn cờ lớn hơn mà không cần tìm kiếm toàn bộ cây trò chơi.

Trong phạm vi thử nghiệm, cấu hình cutoff = 2 có tốc độ nhanh hơn, còn cutoff = 4 cho phép đánh giá các trạng thái ở độ sâu lớn hơn.

### Playtime

Let two heuristic search agents (different cutoff depth, different heuristic evaluation function) compete against each other on a reasonably sized board. Since there is no randomness, you only need to let them play once.

In [19]:
# Heuristic V2: Đánh giá điểm số và nguy cơ trên bàn cờ
def heuristic_eval_v2(board, player=1):
    # Ưu tiên điểm số đã đạt được
    score = (
        sum(v == player for v in board['boxes'].values())
        - sum(v == -player for v in board['boxes'].values())
    )

    # Nếu đã kết thúc, trả về utility chính xác
    if terminal(board):
        return score

    three_sided = 0
    two_sided = 0

    rows, cols = board['size']

    for r in range(rows - 1):
        for c in range(cols - 1):
            if (r, c) in board['boxes']:
                continue

            n = sum(
                edge in board['lines']
                for edge in box_edges(r, c)
            )

            if n == 3:
                three_sided += 1
            elif n == 2:
                two_sided += 1

    # Lượt của người chơi đang được đánh giá
    turn_sign = 1 if board['turn'] == player else -1

    # Ưu tiên cơ hội ghi điểm, xét thêm rủi ro tiềm năng
    value = (
        2.0 * score
        + 0.8 * turn_sign * three_sided
        - 0.15 * turn_sign * two_sided
    )

    return value


# Agent sử dụng heuristic V2 với cutoff = 4
def heuristic_v2_player(board, player=None):
    return alpha_beta_search(
        board,
        player=player,
        cutoff=4,
        ordering=True,
        heuristic=heuristic_eval_v2
    )['move']

In [20]:
match1=play_game(heuristic2_player,heuristic4_player,rows=3,cols=4)
match2=play_game(heuristic4_player,heuristic2_player,rows=3,cols=4)
print('Heuristic2 (+1) vs Heuristic4 (-1), utility:',utility(match1,1))
print('Heuristic4 (+1) vs Heuristic2 (-1), utility:',utility(match2,1))
show_board(match1)
show_board(match2)

Heuristic2 (+1) vs Heuristic4 (-1), utility: -2
Heuristic4 (+1) vs Heuristic2 (-1), utility: 0
•───•───•───•
│ -1│ -1│ -1│
•───•───•───•
│  1│  1│ -1│
•───•───•───•
Score: {1: 2, -1: 4} Turn: -1
•───•───•───•
│  1│  1│  1│
•───•───•───•
│ -1│ -1│ -1│
•───•───•───•
Score: {1: 3, -1: 3} Turn: -1


**Discussion**

Khi Heuristic 2 đi trước và Heuristic 4 đi sau, trận đấu kết thúc với tỷ số 2–4, Heuristic 4 giành chiến thắng.

Khi đổi vị trí để Heuristic 4 đi trước và Heuristic 2 đi sau, trận đấu kết thúc với tỷ số hòa 3–3.

Trong hai ván thử nghiệm, Heuristic 4 đạt một trận thắng và một trận hòa. Mặc dù thuật toán tìm kiếm sâu hơn, kết quả còn phụ thuộc vào trạng thái bàn cờ và thứ tự đi trước. Vì vậy, chưa đủ cơ sở để khẳng định Heuristic 4 luôn mạnh hơn Heuristic 2 hoặc xác định độ sâu tìm kiếm là nguyên nhân trực tiếp tạo ra sự khác biệt.

In [21]:
# So sánh hai agent khác độ sâu và hàm heuristic
# Trận 1: Heuristic 2 đi trước
match_v2_1 = play_game(
    heuristic2_player,
    heuristic_v2_player,
    rows=3,
    cols=4
)

# Trận 2: Heuristic V2 đi trước
match_v2_2 = play_game(
    heuristic_v2_player,
    heuristic2_player,
    rows=3,
    cols=4
)

print("=== Heuristic 2 vs Heuristic V2 ===")

print(
    "Trận 1 - Heuristic 2 (+1) vs V2 (-1):",
    utility(match_v2_1, 1)
)

print(
    "Trận 2 - Heuristic V2 (+1) vs 2 (-1):",
    utility(match_v2_2, 1)
)

print("\nBàn cờ kết thúc trận 1:")
show_board(match_v2_1)

print("\nBàn cờ kết thúc trận 2:")
show_board(match_v2_2)

=== Heuristic 2 vs Heuristic V2 ===
Trận 1 - Heuristic 2 (+1) vs V2 (-1): -6
Trận 2 - Heuristic V2 (+1) vs 2 (-1): 0

Bàn cờ kết thúc trận 1:
•───•───•───•
│ -1│ -1│ -1│
•───•───•───•
│ -1│ -1│ -1│
•───•───•───•
Score: {1: 0, -1: 6} Turn: -1

Bàn cờ kết thúc trận 2:
•───•───•───•
│  1│  1│  1│
•───•───•───•
│ -1│ -1│ -1│
•───•───•───•
Score: {1: 3, -1: 3} Turn: -1


**Discussion: So sánh Heuristic 2 và Heuristic V2**

Trong thí nghiệm này, Heuristic 2 sử dụng cutoff = 2 và hàm đánh giá ban đầu, trong khi Heuristic V2 sử dụng cutoff = 4 và hàm đánh giá cải tiến có bổ sung yếu tố các ô vuông đã có hai cạnh.

Ở trận thứ nhất, khi Heuristic 2 đi trước và Heuristic V2 đi sau, Heuristic V2 giành chiến thắng với tỷ số 6–0, tương ứng giá trị utility = -6 từ góc nhìn người chơi +1.

Ở trận thứ hai, khi Heuristic V2 đi trước và Heuristic 2 đi sau, hai tác tử hòa với tỷ số 3–3, tương ứng utility = 0.

Kết quả cho thấy Heuristic V2 đạt một trận thắng và một trận hòa trong hai ván kiểm thử. Điều này cho thấy cấu hình mới hoạt động hiệu quả trong các tình huống được thử nghiệm.

Tuy nhiên, do Heuristic V2 thay đổi đồng thời cả độ sâu tìm kiếm và hàm đánh giá, chưa thể xác định yếu tố nào đóng góp nhiều hơn vào kết quả. Ngoài ra, chỉ có hai ván đấu từ bàn cờ trống nên chưa đủ cơ sở khẳng định Heuristic V2 luôn mạnh hơn Heuristic 2.

### Kết luận

Trong bài thực hành Dots and Boxes, chương trình đã xây dựng được môi trường trò chơi, các hàm xử lý trạng thái, Random Agent, Alpha-Beta Agent và Heuristic Alpha-Beta Agent.

Kết quả thực nghiệm cho thấy Alpha-Beta có khả năng lựa chọn nước đi tối ưu trên các bàn cờ nhỏ, nhưng số nút duyệt tăng nhanh khi kích thước bàn cờ lớn hơn.

Move ordering giúp cải thiện hiệu quả cắt tỉa bằng cách ưu tiên những nước đi có khả năng ghi điểm, trong khi heuristic kết hợp cutoff giúp giảm chi phí tìm kiếm.

Tuy nhiên, việc giới hạn độ sâu tạo ra sự đánh đổi giữa tốc độ và chất lượng quyết định. Vì vậy, đối với những bàn cờ lớn, cần sử dụng các hàm đánh giá phù hợp và chiến lược tìm kiếm hiệu quả hơn để nâng cao khả năng thi đấu của tác tử.

## Tournament task [+1 to 5% bonus on your course grade; will be assigned separately]

Find another student and let your best agent play against the other student's best player. You are allowed to use any improvements you like as long as you code it yourself. We will set up a class tournament on Canvas. This tournament will continue after the submission deadline.

## Graduate student advanced task: Pure Monte Carlo Search and Best First Move [10 point]

__Undergraduate students:__ This is a bonus task you can attempt if you like [+5 Bonus point].

### Pure Monte Carlo Search

Implement Pure Monte Carlo Search (see [tic-tac-toe-example](tictactoe_pure_monte_carlo_search.ipynb)) and investigate how this search performs on the test boards that you have used above. 

In [17]:
# Your code/ answer goes here.

### Best First Move

How would you determine what the best first move for a standard board ($5 \times 5$) is? You can use Pure Monte Carlo Search or any algorithms that you have implemented above.

In [18]:
# Your code/ answer goes here.